<!--
Copyright Advanced Micro Devices, Inc., or its affiliates.
SPDX-License-Identifier: MIT
-->

# Layouts from scratch — a `pymosaic` tutorial

This notebook builds the idea of a **layout** up from nothing. By the end
you'll be able to *read* any layout at a glance and *compose* your own
to express custom data tilings and GPU schedules.

A layout is one deceptively simple thing:

> **a function that maps a coordinate to an integer offset.**

That's it. `A[i, j] -> some integer`. Everything else — row-major,
column-major, tiling, blocking, swizzles — is just a *choice of that
function*. The magic is that the function is described by two small
tuples (a **shape** and a **stride**) and composes algebraically.

We'll go step by step:

1. What a layout actually *is* (shape + stride → offset)
2. **Stride** is the whole game: row-major vs column-major
3. The offset formula (a dot product)
4. `size` vs `cosize` — domain vs codomain
5. Coordinates ↔ flat indices
6. **Composing** layouts by nesting
7. **Inspecting & reshaping** — pulling layouts apart by mode (and `coalesce`)
8. **Slicing & dicing** — dropping or keeping modes (incl. hierarchical)
9. **Tiling** — building your own schedule with `blocked_product`
10. Your turn: design a layout

Throughout, we lean on `Layout.pretty_print()` so you can *see* every
layout as a grid of offsets.

## Setup

`pymosaic` JIT-compiles a tiny C++ extension the first time you evaluate
a layout of a new *structure*, so the **first** cell that calls into a
layout may take ~1–2 s. After that everything is cached and instant.

The cell below imports `pymosaic`. If you haven't `pip install`ed it, it
also adds the in-repo source checkout (`python/pymosaic`) to the path so
the notebook works straight from a clone.

> Requirements: a C++20 compiler (`g++` by default; set `$CXX` to
> override) plus the `ninja` and `nanobind` Python packages (pulled in
> automatically by `pip install -e python/pymosaic`).

In [1]:
import pathlib
import sys


def _ensure_pymosaic():
    """Import pymosaic, robust to three failure modes:

    1. *Namespace shadow* — if ``repo/python`` is on the path,
       ``import pymosaic`` can resolve to an empty namespace folder with
       no ``Layout``.
    2. *Not installed* — fall back to the in-repo source checkout
       (``python/pymosaic``).
    3. *Stale kernel* — a long-lived kernel that imported ``pymosaic``
       *before* the API grew keeps the old module cached in
       ``sys.modules``, so re-running this cell would otherwise hand back
       a ``Layout`` missing the newer methods.

    We treat ``Layout.dice`` (the newest method this notebook uses) as a
    recency canary: if it's absent we purge ``pymosaic`` from
    ``sys.modules`` and reload from source, so a stale kernel self-heals
    on re-run (no manual restart needed).
    """

    def _fresh_enough(mod) -> bool:
        return hasattr(mod, "Layout") and hasattr(mod.Layout, "dice")

    try:
        import pymosaic

        if _fresh_enough(pymosaic):
            return pymosaic
    except ImportError:
        pass

    # Make sure the in-repo source package is importable.
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        pkg = cand / "python" / "pymosaic"
        if (pkg / "pymosaic" / "__init__.py").exists():
            sys.path.insert(0, str(pkg))
            break

    # Drop any partial/namespace OR stale 'pymosaic' so the next import
    # re-executes the (possibly just-edited) source from disk.
    for name in [m for m in list(sys.modules) if m == "pymosaic" or m.startswith("pymosaic.")]:
        del sys.modules[name]

    import pymosaic

    return pymosaic


pymosaic = _ensure_pymosaic()
from pymosaic import Layout

print("pymosaic", pymosaic.__version__, "ready")

pymosaic 0.1.0 ready


## 1. What is a layout?

A layout has two parts:

- **shape** — how many valid coordinates there are along each axis.
  `(4, 8)` means "4 rows, 8 columns": a domain of 32 coordinates.
- **stride** — how far the offset jumps when you take *one step* along
  each axis.

Let's start with the simplest possible layout: a 1-D run of 8 elements.
The shape is `(8,)`. We didn't give a stride, so `pymosaic` picks the
compact one, `(1,)`: stepping one element along the axis moves the
offset by 1. The layout is the identity `L[i] = i`.

In [2]:
L = Layout(shape=(8,))
print("shape :", L.shape)
print("stride:", L.stride)
print("L[3]  :", L[3])      # the offset at coordinate 3

# pretty_print draws the offset each coordinate maps to.
L.pretty_print()

shape : (8,)
stride: (1,)
L[3]  : 3
(8):(1)   size=8  cosize=8
   0   1   2   3   4   5   6   7
 +---+---+---+---+---+---+---+---+
 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
 +---+---+---+---+---+---+---+---+


## 2. Stride is the whole game

Go up to 2-D, shape `(4, 8)`. There are 32 coordinates `(r, c)`. The
*stride* decides how those 32 coordinates get laid out in a flat buffer.

Two classic choices:

- **Row-major** `stride = (8, 1)`: stepping one column (`c += 1`) moves
  the offset by **1**; stepping one row (`r += 1`) jumps by **8** (a
  whole row). Memory fills left-to-right, top-to-bottom. This is C / NumPy
  default order.
- **Column-major** `stride = (1, 4)`: stepping one row moves the offset
  by **1**; stepping one column jumps by **4** (a whole column). Memory
  fills top-to-bottom, left-to-right. This is Fortran / MATLAB order.

`pymosaic` gives you both as constructors. Watch how the *same shape*
produces two completely different offset grids — that's stride at work.

In [3]:
R = Layout.row_major(4, 8)
print("row-major stride:", R.stride, "  (step a column -> +1, step a row -> +8)")
R.pretty_print()

row-major stride: (8, 1)   (step a column -> +1, step a row -> +8)
(4,8):(8,1)   size=32  cosize=32
     0    1    2    3    4    5    6    7
  +----+----+----+----+----+----+----+----+
0 |  0 |  1 |  2 |  3 |  4 |  5 |  6 |  7 |
  +----+----+----+----+----+----+----+----+
1 |  8 |  9 | 10 | 11 | 12 | 13 | 14 | 15 |
  +----+----+----+----+----+----+----+----+
2 | 16 | 17 | 18 | 19 | 20 | 21 | 22 | 23 |
  +----+----+----+----+----+----+----+----+
3 | 24 | 25 | 26 | 27 | 28 | 29 | 30 | 31 |
  +----+----+----+----+----+----+----+----+


In [4]:
C = Layout.column_major(4, 8)
print("col-major stride:", C.stride, "  (step a row -> +1, step a column -> +4)")
C.pretty_print()

# Same coordinate, different offset — the only thing that changed is stride.
print("\nR[1, 2] =", R[1, 2], " vs  C[1, 2] =", C[1, 2])

col-major stride: (1, 4)   (step a row -> +1, step a column -> +4)
(4,8):(1,4)   size=32  cosize=32
     0    1    2    3    4    5    6    7
  +----+----+----+----+----+----+----+----+
0 |  0 |  4 |  8 | 12 | 16 | 20 | 24 | 28 |
  +----+----+----+----+----+----+----+----+
1 |  1 |  5 |  9 | 13 | 17 | 21 | 25 | 29 |
  +----+----+----+----+----+----+----+----+
2 |  2 |  6 | 10 | 14 | 18 | 22 | 26 | 30 |
  +----+----+----+----+----+----+----+----+
3 |  3 |  7 | 11 | 15 | 19 | 23 | 27 | 31 |
  +----+----+----+----+----+----+----+----+

R[1, 2] = 10  vs  C[1, 2] = 9


## 3. The offset formula

There's no mystery to how a layout turns a coordinate into an offset.
It's a **dot product** of the coordinate with the stride:

$$\text{offset}(c_0, c_1, \dots) = \sum_i c_i \cdot \text{stride}_i$$

For row-major `(4, 8)` with stride `(8, 1)`:

$$\text{offset}(r, c) = r \cdot 8 + c \cdot 1$$

So `R[1, 2] = 1*8 + 2*1 = 10`. Let's confirm the formula reproduces what
the JIT-compiled layout returns, for every coordinate.

In [5]:
shape, stride = R.shape, R.stride   # (4, 8), (8, 1)

ok = True
for r in range(shape[0]):
    for c in range(shape[1]):
        by_hand = r * stride[0] + c * stride[1]
        if by_hand != R[r, c]:
            ok = False
            print(f"mismatch at ({r},{c}): {by_hand} != {R[r, c]}")
print("dot-product formula matches R[r, c] everywhere:", ok)

dot-product formula matches R[r, c] everywhere: True


## 4. `size` vs `cosize` — domain vs codomain

Two numbers describe a layout's "extent":

- **size** = number of valid coordinates = product of the shape. This is
  the size of the **domain** (how many things you can index). In Python
  it's `len(L)`.
- **cosize** = `max(offset) + 1`. This is the span of the **codomain**:
  the smallest buffer that can hold every offset the layout produces.

For a *compact* layout (row- or column-major), `size == cosize`: no gaps,
no overlap. But strides are free — you can make a layout with gaps. A
shape `(4,)` with stride `(2,)` touches offsets `0, 2, 4, 6`: size 4, but
cosize 7. Strided layouts like this are how you describe "every other
element", padded rows, vectorized accesses, and so on.

In [6]:
print(f"compact R(4,8): size={len(R):2d}  cosize={R.cosize:2d}   (no gaps)")

gapped = Layout(shape=(4,), stride=(2,))
print(f"gapped (4,):(2,): size={len(gapped):2d}  cosize={gapped.cosize:2d}   (touches 0,2,4,6)")
gapped.pretty_print()

compact R(4,8): size=32  cosize=32   (no gaps)
gapped (4,):(2,): size= 4  cosize= 7   (touches 0,2,4,6)
(4):(2)   size=4  cosize=7
   0   1   2   3
 +---+---+---+---+
 | 0 | 2 | 4 | 6 |
 +---+---+---+---+


## 5. Coordinates ↔ flat indices

You can index a layout two ways:

- with a full **coordinate**: `R[1, 2]`
- with a single **flat index**: `R[6]`

How does a flat index become a coordinate? By counting through the shape
in **column-major order** (the first axis varies fastest). For shape
`(4, 8)`, flat index `6` decodes to coordinate `(2, 1)` because we count
down the first axis (0,0),(1,0),(2,0),(3,0),(0,1),(1,1),(2,1)... — that's
the 7th coordinate, index 6.

Two helpers (plus indexing) connect the three views — **flat index**,
**coordinate**, and **offset**:

- `idx2crd(i)` — flat index → coordinate (the column-major decode above).
- `crd2idx(coord)` — coordinate → **offset**, applying the strides. This is
  exactly what `L[coord]` does; despite the name it is *not* the inverse of
  `idx2crd`.

So a flat index and the coordinate it decodes to always land on the same
offset: `L[i] == L[idx2crd(i)]`. And `L.enumerate()` walks every coordinate
in that column-major order and applies the layout, giving the full offset
sequence in one C++ loop.

In [7]:
# (1) idx2crd counts column-major through the SHAPE: stepping the flat
#     index by 1 moves DOWN a column (the row advances first). Drawn as a
#     grid, it's just the flat index sitting at each (row, col):
counter = Layout.column_major(*R.shape)   # its offset IS the col-major flat index
print("Flat index at each (row, col)  —  column-major: 0,1,2,3 run DOWN")
print("column 0, then 4 starts column 1 (the row index advances first):")
counter.pretty_print()

print("idx2crd reads a flat index off that grid as a coordinate:")
for i in [0, 1, 4, 6]:
    print(f"   flat {i:2d}  ->  coord {R.idx2crd(i)}")

# (2) crd2idx / R[coord] applies R's strides -> the OFFSET (NOT the inverse
#     of idx2crd). A flat index and its coordinate land on the same offset:
print("\nR (row-major) offset at each (row, col)  —  this grid is R[coord]:")
R.pretty_print()
print(f"check  R[6] = {R[6]}  ==  R[idx2crd(6)] = R[{R.idx2crd(6)}] = {R[R.idx2crd(6)]}")
print(f"       crd2idx({R.idx2crd(6)}) = {R.crd2idx(R.idx2crd(6))}  (the offset, same as R[6])")

# (3) enumerate() reads the offset grid in counting order (flat 0,1,2,...):
print("\nenumerate() — R's offsets in column-major order:")
print(R.enumerate())

Flat index at each (row, col)  —  column-major: 0,1,2,3 run DOWN
column 0, then 4 starts column 1 (the row index advances first):
(4,8):(1,4)   size=32  cosize=32
     0    1    2    3    4    5    6    7
  +----+----+----+----+----+----+----+----+
0 |  0 |  4 |  8 | 12 | 16 | 20 | 24 | 28 |
  +----+----+----+----+----+----+----+----+
1 |  1 |  5 |  9 | 13 | 17 | 21 | 25 | 29 |
  +----+----+----+----+----+----+----+----+
2 |  2 |  6 | 10 | 14 | 18 | 22 | 26 | 30 |
  +----+----+----+----+----+----+----+----+
3 |  3 |  7 | 11 | 15 | 19 | 23 | 27 | 31 |
  +----+----+----+----+----+----+----+----+
idx2crd reads a flat index off that grid as a coordinate:
   flat  0  ->  coord (0, 0)
   flat  1  ->  coord (1, 0)
   flat  4  ->  coord (0, 1)
   flat  6  ->  coord (2, 1)

R (row-major) offset at each (row, col)  —  this grid is R[coord]:
(4,8):(8,1)   size=32  cosize=32
     0    1    2    3    4    5    6    7
  +----+----+----+----+----+----+----+----+
0 |  0 |  1 |  2 |  3 |  4 |  5 |  6 |

## 6. Composing layouts: nesting

Here's the leap that makes layouts powerful: **a mode can itself be a
layout**. Instead of a flat shape `(4, 8)`, a mode can be a *tuple*, like
`((2, 2), 3)`. This is a **hierarchical** (nested) layout, and it's how
you express tiles-within-tiles.

The rule: the **shape and stride must have the same nesting profile**.
If mode 0 of the shape is a 2-tuple `(2, 2)`, then mode 0 of the stride
must also be a 2-tuple, e.g. `(1, 2)`.

The easiest way to build one is `Layout.from_modes(...)`: each argument
becomes one top-level mode. Below we stack a small `(2, 2)` *inner* tile
under a `(3, 3)` *outer* arrangement of those tiles.

**The spacing matters.** A nested layout is still just `coord → offset`.
If the outer tiles are spaced so they don't overlap, the result is a
*bijection* (a clean permutation: every offset hit exactly once). The
inner `(2, 2)` tile occupies offsets `0..3`, so its **cosize is 4** — and
that's exactly how far apart consecutive tiles must sit. We give the
outer mode the column-major strides `(1, 3)` **scaled by 4** → `(4, 12)`.

> If you instead reuse the inner's *own* strides for the outer mode (e.g.
> `from_modes(column_major(2,2), row_major(3,3))`), the tiles overlap and
> offsets repeat — a valid layout, but `size` (36) > `cosize` (12), so
> not a permutation. Section 8's `blocked_product` computes the
> non-overlapping spacing for you automatically.

When you `pretty_print` a nested layout, `pymosaic` automatically draws
**double-line boundaries** at the mode edges and labels the axes with the
nested `(i, j)` coordinates — so the structure is visible, not just
implied. Read each column below: it's one `(2, 2)` tile holding four
consecutive offsets, and the next tile starts 4 later.

In [8]:
inner = Layout.column_major(2, 2)         # a 2x2 tile -> offsets 0,1,2,3 (cosize 4)
outer = Layout(shape=(3, 3), stride=(4, 12))  # 3x3 grid of tiles, spaced by 4 = cosize(inner)
H = Layout.from_modes(inner, outer)

print("H        :", H)
print("H.shape  :", H.shape)        # ((2, 2), (3, 3))
print("H.stride :", H.stride)       # ((1, 2), (4, 12))
bijective = sorted(H.enumerate()) == list(range(len(H)))
print(f"size={len(H)}  cosize={H.cosize}  bijective(permutation)={bijective}")
H.pretty_print()

H        : ((2,2),(3,3)):((1,2),(4,12))
H.shape  : ((2, 2), (3, 3))
H.stride : ((1, 2), (4, 12))
size=36  cosize=36  bijective(permutation)=True
((2,2),(3,3)):((1,2),(4,12))   size=36  cosize=36
        (0,0)   (1,0)   (2,0)   (0,1)   (1,1)   (2,1)   (0,2)   (1,2)   (2,2)
      ┌───────┬───────┬───────╥───────┬───────┬───────╥───────┬───────┬───────┐
(0,0) │     0 │     4 │     8 ║    12 │    16 │    20 ║    24 │    28 │    32 │
      ├───────┼───────┼───────╫───────┼───────┼───────╫───────┼───────┼───────┤
(1,0) │     1 │     5 │     9 ║    13 │    17 │    21 ║    25 │    29 │    33 │
      ╞═══════╪═══════╪═══════╬═══════╪═══════╪═══════╬═══════╪═══════╪═══════╡
(0,1) │     2 │     6 │    10 ║    14 │    18 │    22 ║    26 │    30 │    34 │
      ├───────┼───────┼───────╫───────┼───────┼───────╫───────┼───────┼───────┤
(1,1) │     3 │     7 │    11 ║    15 │    19 │    23 ║    27 │    31 │    35 │
      └───────┴───────┴───────╨───────┴───────┴───────╨───────┴───────┴───────┘


In [9]:
# You can also write the nested tuples explicitly. Here mode 0 is a
# (2, 2) tuple and mode 1 is the scalar 3 — a "mixed" nesting profile.
M = Layout(shape=((2, 2), 3), stride=((1, 2), 4))
print("M:", M, " size:", len(M))
M.pretty_print()

M: ((2,2),3):((1,2),4)  size: 12
((2,2),3):((1,2),4)   size=12  cosize=12
        0    1    2
      ┌────┬────┬────┐
(0,0) │  0 │  4 │  8 │
      ├────┼────┼────┤
(1,0) │  1 │  5 │  9 │
      ╞════╪════╪════╡
(0,1) │  2 │  6 │ 10 │
      ├────┼────┼────┤
(1,1) │  3 │  7 │ 11 │
      └────┴────┴────┘


## 7. Inspecting & reshaping layouts

Once a layout is hierarchical, you'll want to pull it apart and put it
back together — *without* manually indexing into nested tuples.
`pymosaic` ships a family of **structural** accessors: pure tuple
operations that rearrange a layout's *mode tree*. They never do
arithmetic on strides and never evaluate the `coordinate → offset` map —
they just **select, reorder, nest, and unnest** modes.

Two groups:

- **Read a mode** — `shape_at(i)`, `stride_at(i)`, `size_at(i)`, and
  `get(i)` (returns that one mode as its own `Layout`).
- **Reshape the tree** — `flatten()` (drop all nesting), `take(i, j)`
  (keep modes `[i, j)`), `select(*idxs)` (pick / reorder), `group(i, j)`
  (nest a range into one mode), and `append` / `prepend` (splice a mode).

Each also has a free-function form matching mosaic's C++ spelling:
`take(L, 0, 2)` is the same as `L.take(0, 2)`.

> **Why these live in pure Python.** A *structural* op's result is fully
> determined by how the modes are *arranged* — replace every stride with
> an opaque symbol and the output is the same shuffle of those symbols.
> Operations that instead depend on the actual stride *values* (like
> `coalesce`, which merges modes only when `stride_{i+1} == stride_i *
> shape_i`) stay in the C++ layer behind the JIT.

In [10]:
# Reading a hierarchical layout: mode 0 is a (2,2) tile, mode 1 is the scalar 3.
H = Layout(shape=((2, 2), 3), stride=((1, 2), 4))
print("H            :", H, "  rank:", H.rank, " depth:", H.depth)

# Read one mode without reaching into nested tuples by hand:
print("shape_at(0)  :", H.shape_at(0), "   stride_at(0):", H.stride_at(0), "   size_at(0):", H.size_at(0))

# Pull mode 0 out as its own Layout (a nested mode keeps its sub-structure):
print("get(0)       :", H.get(0), "   (the inner 2x2 tile, standalone)")

# Drop all nesting -> one flat list of modes, same underlying map:
print("flatten()    :", H.flatten())

H            : ((2,2),3):((1,2),4)   rank: 2  depth: 2
shape_at(0)  : (2, 2)    stride_at(0): (1, 2)    size_at(0): 4
get(0)       : (2,2):(1,2)    (the inner 2x2 tile, standalone)
flatten()    : (2,2,3):(1,2,4)


In [11]:
# Reshaping the mode tree of a flat rank-3 layout.
L = Layout(shape=(2, 3, 4), stride=(1, 2, 6))
print("L              :", L)
print("take(0, 2)     :", L.take(0, 2), "    # keep modes [0, 2): the 2 and the 3")
print("select(2, 0)   :", L.select(2, 0), "    # pick modes 2 then 0 (reorder)")
print("group(0, 2)    :", L.group(0, 2), "  # nest modes [0, 2) into one mode")

# group and flatten are inverses; the free-function form matches the C++ spelling:
roundtrip = pymosaic.flatten(pymosaic.group(L, 0, 2)) == L
print("flatten(group(L, 0, 2)) == L :", roundtrip)

# append / prepend splice a whole mode (here a new length-5 axis):
new_axis = Layout(shape=5, stride=24)
print("append(5:24)   :", L.append(new_axis))
print("prepend(5:24)  :", L.prepend(new_axis))

L              : (2,3,4):(1,2,6)
take(0, 2)     : (2,3):(1,2)     # keep modes [0, 2): the 2 and the 3
select(2, 0)   : (4,2):(6,1)     # pick modes 2 then 0 (reorder)
group(0, 2)    : ((2,3),4):((1,2),6)   # nest modes [0, 2) into one mode
flatten(group(L, 0, 2)) == L : True
append(5:24)   : (2,3,4,5):(1,2,6,24)
prepend(5:24)  : (5,2,3,4):(24,1,2,6)


### Coalescing — reshaping that *does* read the strides

The structural ops above never look at stride **values**. `coalesce` is
their value-dependent cousin: it returns the **minimal-rank** layout with
the *exact same* `coordinate → offset` map, by merging adjacent modes
whenever they're contiguous (`stride_{i+1} == stride_i * shape_i`).

Because the merge decision depends on the actual strides, the arithmetic
runs in the JIT-compiled C++ (mirroring `mosaic::coalesce`), not in the
pure-tuple layer. The contiguity rule has a **visual tell** in the offset
grids below:

- A **genuinely tiled** layout *interleaves* its tiles: read across a row
  and the offsets **jump** at each tile boundary (`0, 1 ║ 4, 5`). Adjacent
  modes aren't laid out contiguously, so the tiles *can't* fuse —
  `coalesce` only un-nests it.
- A **secretly contiguous** layout has each mode pick up exactly where the
  previous one ended (`stride_{i+1} == stride_i · shape_i`). That's the
  condition for every mode to fuse, so `coalesce` collapses it all the way
  down to a single run.

In [12]:
# (1) A GENUINELY TILED layout: a 4x4 matrix stored as four contiguous
# 2x2 tiles. The double lines mark the tile boundaries; read across a row
# and the offsets JUMP (0, 1 | 4, 5) -- that interleaving IS the tiling.
B = Layout(shape=((2, 2), (2, 2)), stride=((2, 8), (1, 4)))
print("A 2x2-tiled 4x4 matrix — each quadrant is one contiguous tile:")
B.pretty_print()
# Same offset map, but adjacent modes aren't contiguous, so coalesce
# can't fuse across the tile jumps — it only drops the nesting:
print("coalesce(B):", B.coalesce(), " (interleaved tiles can't fuse)\n")

# (2) A layout that *looks* multi-dimensional but is SECRETLY CONTIGUOUS:
# each mode picks up where the previous ended, so every mode fuses and it
# collapses to a single run (its natural traversal is just 0,1,2,...).
M = Layout(shape=((2, 2), 3), stride=((1, 2), 4))
print("Secretly contiguous", M, "— modes chain end-to-end:")
M.pretty_print()
C = M.coalesce()
print(f"coalesce(M): {M.flatten()} (flatten) -> {C} (fully fused, minimal rank)")
print("traversal  :", M.enumerate(), "(0,1,2,... -> fully contiguous)")
print("same map   :", C.enumerate() == M.enumerate())

A 2x2-tiled 4x4 matrix — each quadrant is one contiguous tile:
((2,2),(2,2)):((2,8),(1,4))   size=16  cosize=16
        (0,0)   (1,0)   (0,1)   (1,1)
      ┌───────┬───────╥───────┬───────┐
(0,0) │     0 │     1 ║     4 │     5 │
      ├───────┼───────╫───────┼───────┤
(1,0) │     2 │     3 ║     6 │     7 │
      ╞═══════╪═══════╬═══════╪═══════╡
(0,1) │     8 │     9 ║    12 │    13 │
      ├───────┼───────╫───────┼───────┤
(1,1) │    10 │    11 ║    14 │    15 │
      └───────┴───────╨───────┴───────┘
coalesce(B): (2,2,2,2):(2,8,1,4)  (interleaved tiles can't fuse)

Secretly contiguous ((2,2),3):((1,2),4) — modes chain end-to-end:
((2,2),3):((1,2),4)   size=12  cosize=12
        0    1    2
      ┌────┬────┬────┐
(0,0) │  0 │  4 │  8 │
      ├────┼────┼────┤
(1,0) │  1 │  5 │  9 │
      ╞════╪════╪════╡
(0,1) │  2 │  6 │ 10 │
      ├────┼────┼────┤
(1,1) │  3 │  7 │ 11 │
      └────┴────┴────┘
coalesce(M): (2,2,3):(1,2,4) (flatten) -> (12):(1) (fully fused, minimal rank)
traversal  

## 8. Slicing & dicing — dropping or keeping modes

Slicing a layout with `:` keeps an axis; an integer **drops** it. So
`R[:, 3]` means "keep all rows, fix column 3" and returns a fresh,
**lower-rank** layout describing movement *within the kept axis*.

One subtlety worth internalising: the slice carries the kept axis's
shape/stride, but **not** the base offset contributed by the fixed
coordinate. The intuition (straight from CuTe / mosaic):

> the sub-layout describes movement within the kept modes; the base
> offset belongs to the data pointer, not the layout.

So to recover absolute offsets you add the fixed part back yourself.
Below, column 3 of row-major `(4, 8)` has stride 8 and base offset
`3 * 1 = 3`.

In [13]:
col = R[:, 3]                       # keep rows, fix column 3
print("R[:, 3]      :", col, " (shape", col.shape, "stride", col.stride, ")")
print("col offsets  :", list(col))            # 0, 8, 16, 24  (no base offset)

base = 3 * R.stride[1]              # the dropped column's contribution
absolute = [col[i] + base for i in range(len(col))]
print("+ base", base, "  :", absolute)
print("matches R[r,3]:", [R[r, 3] for r in range(4)])

R[:, 3]      : (4):(8)  (shape (4,) stride (8,) )
col offsets  : [0, 8, 16, 24]
+ base 3   : [3, 11, 19, 27]
matches R[r,3]: [3, 11, 19, 27]


### Slicing into nested modes

When a mode is itself a tuple, the slice coordinate can be a tuple too —
this is **hierarchical slicing**. A nested **int** sub-coord fixes that
leaf; a nested `:` keeps it. So you can reach *inside* a tile and keep
some of its axes while dropping others. Kept inner leaves flatten up into
the result (matching mosaic's `slice`).

Take the hierarchical `H = ((2,2),(2,2))`:

- `H[:, (1, 1)]` keeps mode 0 (the whole inner tile) and **fully fixes**
  mode 1.
- `H[(0,:), (1,:)]` keeps the *second* leaf of each mode.

In [14]:
H = Layout(shape=((2, 2), (2, 2)), stride=((1, 2), (4, 8)))
print("Full hierarchical layout H:")
H.pretty_print()

# ':' keeps a mode; a coordinate fixes it. Tuples descend into nested modes.
sub = H[:, (1, 1)]
print(f"\nH[:, (1, 1)] -> {sub}   (keep mode 0; fully fix nested mode 1)")
sub.pretty_print()

sub2 = H[(0, slice(None)), (1, slice(None))]
print(f"\nH[(0,:), (1,:)] -> {sub2}   (keep the ':' leaf inside each mode)")
sub2.pretty_print()

sub3 = H[:, (1, slice(None))]
print(f"\nH[:, (1, :)] -> {sub3}   (keep mode 0 + the ':' leaf of mode 1)")
sub3.pretty_print()

Full hierarchical layout H:
((2,2),(2,2)):((1,2),(4,8))   size=16  cosize=16
        (0,0)   (1,0)   (0,1)   (1,1)
      ┌───────┬───────╥───────┬───────┐
(0,0) │     0 │     4 ║     8 │    12 │
      ├───────┼───────╫───────┼───────┤
(1,0) │     1 │     5 ║     9 │    13 │
      ╞═══════╪═══════╬═══════╪═══════╡
(0,1) │     2 │     6 ║    10 │    14 │
      ├───────┼───────╫───────┼───────┤
(1,1) │     3 │     7 ║    11 │    15 │
      └───────┴───────╨───────┴───────┘

H[:, (1, 1)] -> ((2,2)):((1,2))   (keep mode 0; fully fix nested mode 1)
((2,2)):((1,2))   size=4  cosize=4
   0   1   2   3
 +---+---+---+---+
 | 0 | 1 | 2 | 3 |
 +---+---+---+---+

H[(0,:), (1,:)] -> (2,2):(2,8)   (keep the ':' leaf inside each mode)
(2,2):(2,8)   size=4  cosize=11
     0    1
  +----+----+
0 |  0 |  8 |
  +----+----+
1 |  2 | 10 |
  +----+----+

H[:, (1, :)] -> ((2,2),2):((1,2),8)   (keep mode 0 + the ':' leaf of mode 1)
((2,2),2):((1,2),8)   size=8  cosize=12
        0    1
      ┌────┬────┐
(0,0) 

### `dice` — keep the fixed modes

`dice` is the exact complement of slicing. Where a slice **keeps** the
`:` axes and drops the fixed ones, `dice` **keeps the fixed axes** and
drops the `:` ones. Same coord, opposite halves — together they
partition the layout's modes.

It's spelled `L.dice(coord)` (or the free function `dice(coord, L)`),
with `coord` written just like an index tuple.

In [15]:
R = Layout.row_major(4, 8)
print("Full layout R (4x8 row-major):")
R.pretty_print()

coord = (slice(None), 3)             # ':' for mode 0, fix mode 1 at column 3
keep = R[coord]                      # slice  -> keep the ':' mode (the rows)
comp = R.dice(coord)                 # dice   -> keep the fixed mode (the column dim)

print(f"\nR[:, 3]   (slice) -> {keep}   keeps the rows (stride 8):")
keep.pretty_print()
print(f"\nR.dice(...) (dice) -> {comp}   keeps the other axis (stride 1):")
comp.pretty_print()

# Slice and dice partition the modes: every axis lands in exactly one half.
print("\ncomplementary halves:", keep.shape, "+", comp.shape, "= modes of", R.shape)

Full layout R (4x8 row-major):
(4,8):(8,1)   size=32  cosize=32
     0    1    2    3    4    5    6    7
  +----+----+----+----+----+----+----+----+
0 |  0 |  1 |  2 |  3 |  4 |  5 |  6 |  7 |
  +----+----+----+----+----+----+----+----+
1 |  8 |  9 | 10 | 11 | 12 | 13 | 14 | 15 |
  +----+----+----+----+----+----+----+----+
2 | 16 | 17 | 18 | 19 | 20 | 21 | 22 | 23 |
  +----+----+----+----+----+----+----+----+
3 | 24 | 25 | 26 | 27 | 28 | 29 | 30 | 31 |
  +----+----+----+----+----+----+----+----+

R[:, 3]   (slice) -> (4):(8)   keeps the rows (stride 8):
(4):(8)   size=4  cosize=25
    0    1    2    3
 +----+----+----+----+
 |  0 |  8 | 16 | 24 |
 +----+----+----+----+

R.dice(...) (dice) -> (8):(1)   keeps the other axis (stride 1):
(8):(1)   size=8  cosize=8
   0   1   2   3   4   5   6   7
 +---+---+---+---+---+---+---+---+
 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
 +---+---+---+---+---+---+---+---+

complementary halves: (4,) + (8,) = modes of (4, 8)


## 9. Tiling — build your own schedule

Now the payoff. Suppose you have a 4×4 grid of work items (GPU thread
blocks, image patches, whatever). The default row-major schedule numbers
them left-to-right, so items `0` and `1` are neighbours but `0` and `4`
are a whole row apart — bad for locality.

We'd rather **tile**: number items so that `0,1,2,3` all fall inside the
top-left 2×2 tile before moving on. That's a *composition* of two
layouts:

- **inner** — the ordering *within* one tile
- **outer** — how the tiles are arranged across the grid

`blocked_product(inner, outer)` stitches them together: it drops a copy
of `inner` at every position of `outer`, offsetting each copy by
`cosize(inner)` so they never collide. The result is a hierarchical
layout — exactly the nested structure from section 6 — that you can
visualize and the tile boundaries pop right out.

`blocked_product` lives in mosaic's C++ algebra; we inline its ~10-line
Python equivalent here (it just builds the nested shape/stride and hands
back an ordinary `Layout`).

In [16]:
def blocked_product(inner: Layout, outer: Layout) -> Layout:
    """Place a copy of `inner` at every position of `outer`.

    Each outer step advances by cosize(inner) so the blocks don't
    collide. Mirrors mosaic::blocked_product (include/mosaic/algebra.hpp).
    """
    n = inner.cosize
    shape = tuple((inner.shape[i], outer.shape[i]) for i in range(inner.rank))
    stride = tuple((inner.stride[i], n * outer.stride[i]) for i in range(inner.rank))
    return Layout(shape=shape, stride=stride)


inner = Layout.row_major(2, 2)       # ordering within a 2x2 tile
outer = Layout.column_major(2, 2)    # how the four tiles are placed
tiled = blocked_product(inner, outer)

print("tiled:", tiled, " size:", len(tiled))
tiled.pretty_print()

# A valid schedule visits every position exactly once (a permutation).
mapped = [tiled[i] for i in range(len(tiled))]
print("\nis a permutation of [0, 16):", sorted(mapped) == list(range(len(tiled))))

tiled: ((2,2),(2,2)):((2,4),(1,8))  size: 16
((2,2),(2,2)):((2,4),(1,8))   size=16  cosize=16
        (0,0)   (1,0)   (0,1)   (1,1)
      ┌───────┬───────╥───────┬───────┐
(0,0) │     0 │     1 ║     8 │     9 │
      ├───────┼───────╫───────┼───────┤
(1,0) │     2 │     3 ║    10 │    11 │
      ╞═══════╪═══════╬═══════╪═══════╡
(0,1) │     4 │     5 ║    12 │    13 │
      ├───────┼───────╫───────┼───────┤
(1,1) │     6 │     7 ║    14 │    15 │
      └───────┴───────╨───────┴───────┘

is a permutation of [0, 16): True


## 10. Your turn — design a layout

Everything you've seen composes. Try building your own schedule by
changing the four knobs below and re-running the cell:

- the **tile size** (`inner`) — how items are ordered inside one tile
- the **grid of tiles** (`outer`) — how tiles are laid across the space
- swap `row_major` ↔ `column_major` on either one to flip the ordering

The cell prints the layout, checks it's a valid permutation, and draws
the tiles. A few ideas to try:

- `inner = column_major(2, 2)` — fill each tile top-to-bottom
- `outer = row_major(2, 4)` — sweep tiles across rows first
- bump the tile to `4, 4` for a coarser blocking

Watch where the double-line boundaries land and how the numbers flow.

In [17]:
# ---- knobs: change these and re-run ----
inner = Layout.row_major(4,4)       # ordering within a tile
outer = Layout.column_major(2, 2)    # arrangement of tiles  (grid = 6 x 4)
# -----------------------------------------

my_layout = blocked_product(inner, outer)
mapped = [my_layout[i] for i in range(len(my_layout))]

print("my_layout:", my_layout)
print("grid size:", len(my_layout), " permutation:", sorted(mapped) == list(range(len(my_layout))))
my_layout.pretty_print()

my_layout: ((4,2),(4,2)):((4,16),(1,32))
grid size: 64  permutation: True
((4,2),(4,2)):((4,16),(1,32))   size=64  cosize=64
        (0,0)   (1,0)   (2,0)   (3,0)   (0,1)   (1,1)   (2,1)   (3,1)
      ┌───────┬───────┬───────┬───────╥───────┬───────┬───────┬───────┐
(0,0) │     0 │     1 │     2 │     3 ║    32 │    33 │    34 │    35 │
      ├───────┼───────┼───────┼───────╫───────┼───────┼───────┼───────┤
(1,0) │     4 │     5 │     6 │     7 ║    36 │    37 │    38 │    39 │
      ├───────┼───────┼───────┼───────╫───────┼───────┼───────┼───────┤
(2,0) │     8 │     9 │    10 │    11 ║    40 │    41 │    42 │    43 │
      ├───────┼───────┼───────┼───────╫───────┼───────┼───────┼───────┤
(3,0) │    12 │    13 │    14 │    15 ║    44 │    45 │    46 │    47 │
      ╞═══════╪═══════╪═══════╪═══════╬═══════╪═══════╪═══════╪═══════╡
(0,1) │    16 │    17 │    18 │    19 ║    48 │    49 │    50 │    51 │
      ├───────┼───────┼───────┼───────╫───────┼───────┼───────┼───────┤
(1,1) │    20

## Recap

You now have the whole mental model:

| Concept | One-liner |
| --- | --- |
| **layout** | a function `coordinate -> offset` |
| **shape** | how many coordinates per axis (the domain) |
| **stride** | how far the offset steps per axis |
| **offset** | `sum(coord_i * stride_i)` — a dot product |
| **size / cosize** | domain count / codomain span (buffer size) |
| **idx2crd** | flat index → coordinate (column-major counting) |
| **crd2idx** | coordinate → offset (applies strides; same as `L[coord]`) |
| **nesting** | a mode can be a layout → tiles within tiles |
| **structural ops** | `get`/`take`/`select`/`group`/`flatten`/`append` — reshape the mode tree (pure tuples, no stride math) |
| **coalesce** | merge contiguous modes → minimal-rank, same map (value-aware, runs in C++) |
| **slicing** | drop a mode (incl. nested coords) to get a lower-rank sub-layout |
| **dice** | keep the fixed modes — the complement of slicing |
| **blocked_product** | compose inner + outer into a tiling |

Everything beyond this — complement, swizzles, GPU thread-value
layouts — is built from these same pieces.

### Where to go next

- `01_basics.py` … `05_cache_inspection.py` — the runnable script tutorials
  in this folder, in the same order.
- `04_tiling.py` — the tiling story as a standalone script.
- `basic_tiling/`, `mfma/`, `gemm_grid/`, `slice/` — ports of the C++
  examples showing real GPU schedules.
- `python/pymosaic/pymosaic/pretty.py` — the visualizer used throughout.